# **002 — PREPROCESSING (ESA-ADB)**

Built from scratch, following **Kotowski et al., "European Space Agency Dataset and
Benchmark for Real-World Anomaly Detection in Spacecraft Time Series"** (ESA-ADB), and
cross-checked line-by-line against the authors' own reference implementation
(`kplabs-pl/ESA-ADB`), not just the paper's prose description.

## Scope of this notebook

- Mission1, lightweight channel subset (channels 41–46) first — this is the ESA-ADB
  authors' own designated fast-iteration subset, and it's what Table 4 in the paper
  reports scores for, so we have something concrete to validate against.
- Switching to Mission2 or to the full channel set later is a one-line config change
  (see the CONFIG cell) — the functions themselves are mission-agnostic.
- **What this notebook deliberately does NOT do:** standardization (z-scoring) and
  window labeling. In the reference implementation, standardization happens
  per-algorithm at train time (using nominal-only training statistics), not baked
  into the shared preprocessed file. We follow that same separation of concerns —
  it keeps this file a single source of truth that every later modeling notebook
  can reuse without re-deriving it differently each time.


# **1. IMPORT**

In [ ]:
import json
from enum import IntEnum
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive
print(f"pandas version in this environment: {pd.__version__}")


pandas version in this environment: 2.2.2


# **2. MOUNT GOOGLE DRIVE(IT IS PROPOSED TO RUN THIS NOTEBOOK IN GOOGLE COLAB AND USE GOOGLE DRIVE TO STORE DATA)**



In [ ]:
drive.mount('/content/drive')
DRIVE_ROOT = Path("/content/drive/MyDrive/BeaconProject")

Mounted at /content/drive
IN_COLAB = True
DRIVE_ROOT = /content/drive/MyDrive/BeaconProject


# **3. CONFIG**
Everything mission-specific lives here. Resampling rule, monotonic channel
ranges, and train/test split dates are taken directly from the reference
implementation's own preprocessing scripts for Mission1 and Mission2 — not
guessed from the paper text alone.

**Path layout, matching your actual Drive folder:**
`BeaconProject/ESA-Mission1/ESA-Mission1/channels/channel_41/channel_41`
— note the mission folder appears twice (that's just the zip's internal root
folder carrying over when it was extracted) and each channel is its own folder
containing a same-named file with no extension (that's what happens when a
`channel_41.zip` gets extracted rather than kept zipped). `channel_pickle_path()`
below builds exactly that path; `load_raw_pickle()` tries reading it a couple of
different ways since we can't be 100% sure whether Drive's extraction left the
contents still zip-compressed internally or not.


In [ ]:
MISSION_CONFIG = {
    "ESA-Mission1": {
        "resampling_rule": pd.Timedelta(seconds=30),       # paper: 0.033 Hz
        "monotonic_channel_range": (4, 11),                # differentiate these before use
        "test_data_split": "2007-01-01",                   # everything from here on = test set
        "val_split": "2006-10-01",                          # last 3 months of train = validation
        "lightweight_channels": [f"channel_{i}" for i in range(41, 47)],  # 41-46
    },
    "ESA-Mission2": {
        "resampling_rule": pd.Timedelta(seconds=18),       # paper: 0.056 Hz
        "monotonic_channel_range": (29, 46),
        "test_data_split": "2001-10-01",
        "val_split": "2001-07-01",
        "lightweight_channels": [f"channel_{i}" for i in range(18, 29)],  # 18-28
    },
}

# --- EDIT THIS ONE LINE to switch missions ---
ACTIVE_MISSION = "ESA-Mission1"

# doubled folder name matches your actual Drive structure
RAW_DATA_ROOT = DRIVE_ROOT / ACTIVE_MISSION / ACTIVE_MISSION

# write locally first (fast; Colab's local disk is wiped when the session ends),
# then a later cell copies the finished files to Drive to persist them
OUTPUT_DIR_LOCAL = Path("/content/beacon_preprocessed" if IN_COLAB else "data/preprocessed") / ACTIVE_MISSION
OUTPUT_DIR_DRIVE = DRIVE_ROOT / "preprocessed" / ACTIVE_MISSION

CFG = MISSION_CONFIG[ACTIVE_MISSION]
TARGET_CHANNELS = CFG["lightweight_channels"]
RESAMPLING_RULE = CFG["resampling_rule"]
INCLUDE_TELECOMMANDS = False   # simple baselines (GlobalSTD/PCC/etc.) don't consume these; flip on later for Telemanom-ESA/DC-VAE-ESA
MIN_TELECOMMAND_PRIORITY = 3   # only used if INCLUDE_TELECOMMANDS = True


def channel_pickle_path(channel_name: str) -> Path:
    return RAW_DATA_ROOT / "channels" / channel_name / channel_name


def telecommand_pickle_path(tc_name: str) -> Path:
    return RAW_DATA_ROOT / "telecommands" / tc_name / tc_name


def load_raw_pickle(path: Path) -> pd.DataFrame:
    '''Try a few compression settings - an unzipped Drive folder usually means
    a plain pickle (compression=None), but this falls back gracefully if not.'''
    last_err = None
    for compression in (None, "zip", "infer"):
        try:
            return pd.read_pickle(path, compression=compression)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Could not read {path} with any compression setting. Last error: {last_err}")


OUTPUT_DIR_LOCAL.mkdir(parents=True, exist_ok=True)
print(f"Active mission: {ACTIVE_MISSION}")
print(f"Target channels ({len(TARGET_CHANNELS)}): {TARGET_CHANNELS}")
print(f"Resampling rule: {RESAMPLING_RULE}")
print(f"Raw data expected at: {RAW_DATA_ROOT}")
print(f"Example channel path: {channel_pickle_path(TARGET_CHANNELS[0])}")


Active mission: ESA-Mission1
Target channels (6): ['channel_41', 'channel_42', 'channel_43', 'channel_44', 'channel_45', 'channel_46']
Resampling rule: 0 days 00:00:30
Raw data expected at: /content/drive/MyDrive/BeaconProject/ESA-Mission1/ESA-Mission1
Example channel path: /content/drive/MyDrive/BeaconProject/ESA-Mission1/ESA-Mission1/channels/channel_41/channel_41


# **4. ENVIRONMENT AND LIBRARY VERSION CHECK**
The raw ESA-AD channel/telecommand files were originally pickled with
**pandas 1.5.3** (stated explicitly in the dataset paper, Appendix B.5). Pandas
does not guarantee pickle compatibility across major versions, so if Colab's
pandas has drifted far from that, reading the raw files can fail — or worse,
silently return something subtly wrong. Test this on ONE real file before
trusting anything else.


In [ ]:
_test_channel = TARGET_CHANNELS[0]
_test_path = channel_pickle_path(_test_channel)
print(f"Testing read of: {_test_path}")

if _test_path.exists():
    try:
        _test_df = load_raw_pickle(_test_path)
        print("OK - pickle loaded successfully.")
        print(_test_df.head())
        print(_test_df.dtypes)
    except Exception as e:
        print("FAILED to read the raw pickle.")
        print(f"Error: {e}")
        print("\nIf this fails: double-check the folder actually contains a file named "
              f"'{_test_channel}' with no extension inside it, and that Drive finished syncing.")
else:
    print(f"File not found at {_test_path}.")
    print("Double-check: Drive mounted correctly, ACTIVE_MISSION is right, and the folder "
          "structure really is BeaconProject/<mission>/<mission>/channels/<channel>/<channel>.")


Testing read of: /content/drive/MyDrive/BeaconProject/ESA-Mission1/ESA-Mission1/channels/channel_41/channel_41
OK - pickle loaded successfully.
                         channel_41
datetime                           
2000-01-01 00:00:16.353    0.812578
2000-01-01 00:00:46.353    0.821213
2000-01-01 00:01:08.853    0.808654
2000-01-01 00:01:16.353    0.819642
2000-01-01 00:01:46.353    0.821996
channel_41    float32
dtype: object


# **5. LOADING THE METADATA FILES**
`labels.csv`, `anomaly_types.csv`, `channels.csv`, `telecommands.csv` live directly
inside each mission folder (ESA-AD Appendix B.5, Table 8). We print schemas rather
than assume exact column names/casing — if anything below looks different from
what's printed, that's the signal to adjust, not a bug in the code.


In [ ]:
labels_df = pd.read_csv(RAW_DATA_ROOT / "labels.csv", parse_dates=["StartTime", "EndTime"])
anomaly_types_df = pd.read_csv(RAW_DATA_ROOT / "anomaly_types.csv")
channels_meta_df = pd.read_csv(RAW_DATA_ROOT / "channels.csv")
telecommands_meta_df = pd.read_csv(RAW_DATA_ROOT / "telecommands.csv") if INCLUDE_TELECOMMANDS else None

print("labels.csv columns:", labels_df.columns.tolist())
print("anomaly_types.csv columns:", anomaly_types_df.columns.tolist())
print("channels.csv columns:", channels_meta_df.columns.tolist())
if telecommands_meta_df is not None:
    print("telecommands.csv columns:", telecommands_meta_df.columns.tolist())

print()
print("labels.csv sample:")
display(labels_df.head())
print()
print("anomaly_types.csv sample:")
display(anomaly_types_df.head())


labels.csv columns: ['ID', 'Channel', 'StartTime', 'EndTime']
anomaly_types.csv columns: ['ID', 'Class', 'Subclass', 'Category', 'Dimensionality', 'Locality', 'Length']
channels.csv columns: ['Channel', 'Subsystem', 'Physical Unit', 'Group', 'Target']

labels.csv sample:


,ID,Channel,StartTime,EndTime
0,id_1,channel_12,2004-12-01 20:42:15.429000+00:00,2004-12-08 22:55:45.429000+00:00
1,id_1,channel_13,2004-12-01 20:42:15.429000+00:00,2004-12-08 22:55:45.429000+00:00
2,id_1,channel_14,2004-12-01 20:43:45.429000+00:00,2004-12-02 02:57:15.429000+00:00
3,id_1,channel_15,2004-12-01 20:45:00.429000+00:00,2004-12-02 02:58:45.429000+00:00
4,id_1,channel_16,2004-12-01 20:43:45.429000+00:00,2004-12-16 16:52:30.429000+00:00


anomaly_types.csv sample:


,ID,Class,Subclass,Category,Dimensionality,Locality,Length
0,id_1,class_6,subclass_1,Rare Event,Multivariate,Global,Subsequence
1,id_2,class_7,subclass_1,Anomaly,Multivariate,Local,Subsequence
2,id_3,class_7,subclass_1,Anomaly,Multivariate,Global,Subsequence
3,id_4,class_7,subclass_1,Anomaly,Multivariate,Local,Subsequence
4,id_5,class_7,subclass_1,Anomaly,Multivariate,Local,Subsequence


# **5.1] CROSS-CHECK WHETHER OUR TARGET CHANNELS ARE ACTULLY FLAGGED AS TARGETS ?**
`channels.csv` carries a target/non-target flag per the paper (Appendix B.5,
Appendix A.4). This just confirms `TARGET_CHANNELS` above lines up with what the
metadata says, before we spend time processing anything.


In [ ]:
target_col_candidates = [c for c in channels_meta_df.columns if "target" in c.lower()]
name_col_candidates = [c for c in channels_meta_df.columns if c.lower() in ("channel", "name", "channel_name")]
print(f"Guessed target-flag column: {target_col_candidates}")
print(f"Guessed channel-name column: {name_col_candidates}")

if target_col_candidates and name_col_candidates:
    tcol, ncol = target_col_candidates[0], name_col_candidates[0]
    metadata_targets = set(channels_meta_df.loc[channels_meta_df[tcol].astype(bool), ncol])
    missing = set(TARGET_CHANNELS) - metadata_targets
    if missing:
        print(f"WARNING: these channels are not flagged as targets in channels.csv: {missing}")
    else:
        print("OK - all configured TARGET_CHANNELS are flagged as target channels in the metadata.")
else:
    print("Could not auto-detect the target-flag/name columns — inspect channels_meta_df.columns "
          "above and adjust target_col_candidates/name_col_candidates manually if you want this check.")


Guessed target-flag column: ['Target']
Guessed channel-name column: ['Channel']
OK - all configured TARGET_CHANNELS are flagged as target channels in the metadata.


# **6. EVENT CATEGORIES (PAPER TABLE 5, APPENDIX A.6)**
Four categories exist: `Anomaly`, `Rare Event`, `Communication Gap`,`Invalid
Segment`. Only Anomaly and Rare Event are "real" signal events that a point-preserving
resample must protect — gaps and invalid segments are structural, not point-in-time
signal content, so they don't need the same protection.


In [ ]:
class AnnotationLabel(IntEnum):
    NOMINAL = 0
    ANOMALY = 1
    RARE_EVENT = 2
    GAP = 3
    INVALID = 4

CATEGORY_TO_LABEL = {
    "Anomaly": AnnotationLabel.ANOMALY,
    "Rare Event": AnnotationLabel.RARE_EVENT,
    "Communication Gap": AnnotationLabel.GAP,
    "Invalid Segment": AnnotationLabel.INVALID,
}

# Labels that must never be silently erased by resampling
PRESERVE_LABELS = {AnnotationLabel.ANOMALY, AnnotationLabel.RARE_EVENT}


# **7. STAMP PER-SAMPLE LABELS ONTO THE RAW (IRREGULAR) DATA ~ [STEP - A]**
This happens *before* any resampling, directly on the original timestamps, using
closed-interval assignment (`StartTime:EndTime` inclusive), matching the reference
implementation exactly.


In [ ]:
def assign_point_labels(channel_df: pd.DataFrame, channel_name: str,
                         labels_df: pd.DataFrame, anomaly_types_df: pd.DataFrame) -> pd.DataFrame:
    df = channel_df.copy()
    df["label"] = np.uint8(AnnotationLabel.NOMINAL)
    df = df.sort_index()

    channel_events = labels_df[labels_df["Channel"] == channel_name]
    for _, row in channel_events.iterrows():
        category = anomaly_types_df.loc[anomaly_types_df["ID"] == row["ID"], "Category"].values[0]
        label_value = CATEGORY_TO_LABEL.get(category, AnnotationLabel.GAP)
        df.loc[row["StartTime"]:row["EndTime"], "label"] = label_value
    return df


# **7. ZERO-ORDER-HOLD RESAMPLING (APPENDIX C.3, STEPS 1-2) ~ STEP B**
Build a uniform grid at the mission's target frequency, then forward-fill. The **very**
first grid point is forced to the actual first raw sample (otherwise it would be
`NaN`, since nothing precedes it to forward-fill from).


In [ ]:
def zero_order_hold_resample(raw_df: pd.DataFrame, resampling_rule: pd.Timedelta) -> pd.DataFrame:
    first_grid = pd.Timestamp(raw_df.index[0]).floor(freq=resampling_rule)
    last_grid = pd.Timestamp(raw_df.index[-1]).ceil(freq=resampling_rule)
    grid = pd.date_range(first_grid, last_grid, freq=resampling_rule)

    resampled = raw_df.reindex(grid, method="ffill")
    resampled.iloc[0] = raw_df.iloc[0]
    return resampled


## **7. RESTORE POINT EVENTS ERASED BY RESAMPLING (APPENDIX C.3, STEP 3) ~ STEP C**

Plain forward-fill only keeps whichever raw sample happened to be *last* inside
each grid bin. If a point anomaly appeared mid-bin and the signal recovered to
nominal before the bin ended, the grid point for that bin shows the recovered
(nominal) value — the anomaly is invisible, not just downweighted.

**Fix:** re-bin the *raw* data using the same frequency. For any bin containing
more than one raw sample, if the bin's last raw sample is nominal but an
anomalous/rare-event sample occurred earlier in that same bin, force-write that
sample's value onto the grid point immediately *after* the bin — shifting it
forward by one step rather than losing it. This is exactly what the paper's
Appendix C.3 step 3 describes ("assign its value and label to the latter
timestamp from the pair").

**Performance note:** the reference implementation bins the raw data using
`pd.Grouper(freq=...)`, which enumerates *every* calendar bin across the full
time range — including empty ones. For a multi-year channel at 30-second
resolution that's millions of empty bins to loop over in Python, and it will
silently hang for a very long time on real data. We bin with `.floor()`
instead, which only enumerates bins that actually contain a raw sample —
identical result, far faster. (Measured on a 7.5-year synthetic span: ~7.8
million groups with `pd.Grouper` vs. a few thousand with `.floor()`.)


In [ ]:
def restore_erased_point_events(raw_df: pd.DataFrame, resampled_df: pd.DataFrame,
                                 resampling_rule: pd.Timedelta):
    resampled_df = resampled_df.copy()
    # NOTE: pd.Grouper(freq=...) would enumerate every calendar bin across the
    # full index range, including empty ones -- for a multi-year span at 30s
    # resolution that's millions of empty groups to loop over in Python (this
    # was measured: ~7.8M groups for a ~7.5-year span, and it will silently
    # hang for minutes-to-hours on real data). .floor() groups only the bins
    # that actually contain raw samples -- identical result, orders of
    # magnitude faster.
    bin_labels = raw_df.index.floor(resampling_rule)

    n_restored = 0
    for bin_start, group in raw_df.groupby(bin_labels):
        if len(group) <= 1:
            continue
        if group["label"].iloc[-1] != AnnotationLabel.NOMINAL:
            continue  # bin already ends anomalous -> ffill already captured it correctly
        is_preserved = group["label"].isin([int(l) for l in PRESERVE_LABELS])
        if is_preserved.any():
            target_ts = bin_start + resampling_rule
            last_preserved_row = group[is_preserved].iloc[-1]
            if target_ts in resampled_df.index:
                resampled_df.loc[target_ts] = last_preserved_row
                n_restored += 1
    return resampled_df, n_restored


# **8. CROSS-VERIFICATION - WATCH IT CATCH THE EXACT OLD BUG WHICH OCCURED BEACUSE WE USED FAST-FORWARD FILLING ONLY**
Before trusting this on 11.6GB of real data, here's a deliberately constructed
worst-case example: a single point anomaly hiding in the middle of a resampling
bin, with the signal recovered to nominal by the time the bin ends — precisely
the case that erased anomalies last time.

We compare three things:
1. **Naive resample** (ffill only, no correction) — expected to erase it
2. **Old median-based approach** (what the previous notebook did) — expected to erase it
3. **Corrected pipeline** (this notebook) — expected to preserve it

If any assertion below fails, stop — do not proceed to real data.


In [ ]:
_rule = pd.Timedelta(seconds=30)

_timestamps = pd.to_datetime([
    "2000-01-01 00:00:02",  # nominal
    "2000-01-01 00:00:12",  # <-- POINT ANOMALY, mid-bin
    "2000-01-01 00:00:25",  # recovered to nominal (last sample in the bin)
    "2000-01-01 00:00:35",  # next bin
    "2000-01-01 00:00:55",
])
_values = [10.0, 999.0, 10.1, 10.2, 10.3]
_raw = pd.DataFrame({"value": _values}, index=_timestamps)

_labels = pd.DataFrame({
    "ID": ["demo_evt"], "Channel": ["channel_demo"],
    "StartTime": [pd.to_datetime("2000-01-01 00:00:12")],
    "EndTime": [pd.to_datetime("2000-01-01 00:00:12")],
})
_types = pd.DataFrame({"ID": ["demo_evt"], "Category": ["Anomaly"]})

_labeled_raw = assign_point_labels(_raw, "channel_demo", _labels, _types)
print("Raw (irregular) data with labels:")
display(_labeled_raw)

# 1. naive resample
_naive = zero_order_hold_resample(_labeled_raw, _rule)
naive_has_anomaly = (_naive["label"] == AnnotationLabel.ANOMALY).any()
print("\n1) Naive ffill-only resample:")
display(_naive)
print(f"   Anomaly visible? {naive_has_anomaly}")

# 2. old median-based approach
_median = _labeled_raw.resample(_rule).median()
median_has_anomaly = (_median["value"] == 999.0).any()
print("\n2) Old median-based resample (what the previous notebook did):")
display(_median)
print(f"   Peak (999.0) visible? {median_has_anomaly}")

# 3. corrected pipeline
_corrected, _n = restore_erased_point_events(_labeled_raw, _naive, _rule)
corrected_has_anomaly = (_corrected["label"] == AnnotationLabel.ANOMALY).any()
print("\n3) Corrected pipeline (this notebook):")
display(_corrected)
print(f"   Anomaly visible? {corrected_has_anomaly} (events restored: {_n})")

assert not naive_has_anomaly, "test setup issue: naive resample should have erased this"
assert not median_has_anomaly, "test setup issue: median resample should have erased this"
assert corrected_has_anomaly, "STOP: correction failed to restore the erased point anomaly"
print("\nPASS — both the naive and old median approaches erase the anomaly; the corrected pipeline restores it.")


Raw (irregular) data with labels:


,value,label
2000-01-01 00:00:02,10.0,0
2000-01-01 00:00:12,999.0,1
2000-01-01 00:00:25,10.1,0
2000-01-01 00:00:35,10.2,0
2000-01-01 00:00:55,10.3,0



1) Naive ffill-only resample:


,value,label
2000-01-01 00:00:00,10.0,0.0
2000-01-01 00:00:30,10.1,0.0
2000-01-01 00:01:00,10.3,0.0


   Anomaly visible? False

2) Old median-based resample (what the previous notebook did):


,value,label
2000-01-01 00:00:00,10.10,0.0
2000-01-01 00:00:30,10.25,0.0


   Peak (999.0) visible? False

3) Corrected pipeline (this notebook):


,value,label
2000-01-01 00:00:00,10.0,0.0
2000-01-01 00:00:30,999.0,1.0
2000-01-01 00:01:00,10.3,0.0


   Anomaly visible? True (events restored: 1)

PASS — both the naive and old median approaches erase the anomaly; the corrected pipeline restores it.


# **9. TELECOMMAND ENCODING (OPTIONAL - OFF BY DEFAULT)**
Each execution timestamp is bracketed with zero-valued samples immediately
before/after (± one resampling step) so forward-fill can't smear a single
instantaneous command into a long plateau.


In [ ]:
def encode_telecommand(tc_raw_df: pd.DataFrame, resampling_rule: pd.Timedelta) -> pd.DataFrame:
    df = tc_raw_df.sort_index().copy()
    df = df[~df.index.duplicated()]
    original_timestamps = df.index.copy()

    for ts in original_timestamps:
        before, after = ts - resampling_rule, ts + resampling_rule
        if len(df.loc[before:ts]) == 1:
            df.loc[before] = 0
            df = df.sort_index()  # re-sort before the next .loc slice, or it silently misreads
        if len(df.loc[ts:after]) == 1:
            df.loc[after] = 0
            df = df.sort_index()

    first_grid = pd.Timestamp(df.index[0]).floor(freq=resampling_rule)
    last_grid = pd.Timestamp(df.index[-1]).ceil(freq=resampling_rule)
    grid = pd.date_range(first_grid, last_grid, freq=resampling_rule)
    resampled = df.reindex(grid, method="ffill")
    resampled.iloc[0] = df.iloc[0]
    return resampled


# **10. ORCHESTRATION**
- `process_channel`: labels + resample + correction, per channel
- `build_wide_dataset`: aligns every processed channel (and optional telecommand)
  onto one shared time grid — matching `find_full_time_range` +
  `reindex().ffill().bfill()` from the reference implementation
- `split_train_val_test`: chronological split using the mission's official dates
  (Section 2.3 of the paper: first half of the mission = train, with the last
  3 months of that held out as validation; second half = test)

No window labels are created anywhere here — deliberately.


In [ ]:
def process_channel(raw_df, channel_name, labels_df, anomaly_types_df, resampling_rule):
    labeled_raw = assign_point_labels(raw_df, channel_name, labels_df, anomaly_types_df)
    resampled = zero_order_hold_resample(labeled_raw, resampling_rule)
    corrected, n_restored = restore_erased_point_events(labeled_raw, resampled, resampling_rule)
    corrected = corrected.rename(columns={"value": channel_name, "label": f"is_anomaly_{channel_name}"})
    return corrected, n_restored


def build_wide_dataset(channel_frames: dict, telecommand_frames: dict, resampling_rule):
    all_frames = {**channel_frames, **telecommand_frames}
    start = min(df.index.min() for df in all_frames.values())
    end = max(df.index.max() for df in all_frames.values())
    full_grid = pd.date_range(start, end, freq=resampling_rule)

    wide = pd.DataFrame(index=full_grid)
    for name, df in all_frames.items():
        aligned = df.reindex(full_grid)
        for col in aligned.columns:
            filled = aligned[col].ffill().bfill()
            wide[col] = filled.astype(np.uint8) if col.startswith("is_anomaly_") else filled
    wide.index.name = "timestamp"
    return wide


def split_train_val_test(wide_df, test_start, val_start):
    train_full = wide_df[wide_df.index < pd.to_datetime(test_start)]
    test = wide_df[wide_df.index >= pd.to_datetime(test_start)]
    train = train_full[train_full.index < pd.to_datetime(val_start)]
    val = train_full[train_full.index >= pd.to_datetime(val_start)]
    return train, val, test


# **11. RUN IT ON REAL DATA**
Reads each raw channel pickle, labels it, resamples it with the point-preserving
correction, combines everything onto one shared grid, splits chronologically, and
writes `train.csv` / `val.csv` / `test.csv` plus a metadata JSON documenting exactly
what parameters were used

In [ ]:
# Normalize timezones once, up front. Real ESA pickles are commonly tz-aware
# (e.g. UTC), while labels.csv parses as tz-naive by default via pd.read_csv.
# pandas refuses to compare across that boundary ("Cannot compare tz-naive
# and tz-aware datetime-like objects"), which is exactly what you hit.
if labels_df["StartTime"].dt.tz is not None:
    labels_df["StartTime"] = labels_df["StartTime"].dt.tz_localize(None)
if labels_df["EndTime"].dt.tz is not None:
    labels_df["EndTime"] = labels_df["EndTime"].dt.tz_localize(None)


def _drop_tz(df: pd.DataFrame) -> pd.DataFrame:
    """Strip timezone info from a DatetimeIndex if present, so it can be
    compared against labels_df's now-tz-naive StartTime/EndTime."""
    if isinstance(df.index, pd.DatetimeIndex) and df.index.tz is not None:
        df = df.copy()
        df.index = df.index.tz_localize(None)
    return df


channel_frames = {}
restore_counts = {}

for channel_name in TARGET_CHANNELS:
    path = channel_pickle_path(channel_name)
    raw_df = load_raw_pickle(path)
    raw_df = raw_df.rename(columns={channel_name: "value"})
    raw_df = _drop_tz(raw_df)
    processed, n_restored = process_channel(raw_df, channel_name, labels_df, anomaly_types_df, RESAMPLING_RULE)
    channel_frames[channel_name] = processed
    restore_counts[channel_name] = n_restored
    print(f"{channel_name}: {len(raw_df):>8,} raw samples -> {len(processed):>7,} resampled rows "
          f"| point events restored: {n_restored}")

telecommand_frames = {}
if INCLUDE_TELECOMMANDS:
    priority_col = [c for c in telecommands_meta_df.columns if "priority" in c.lower()][0]
    name_col = [c for c in telecommands_meta_df.columns if c.lower() in ("telecommand", "name")][0]
    high_priority_tcs = telecommands_meta_df.loc[
        telecommands_meta_df[priority_col] >= MIN_TELECOMMAND_PRIORITY, name_col
    ].tolist()
    print(f"\nEncoding {len(high_priority_tcs)} telecommands with priority >= {MIN_TELECOMMAND_PRIORITY}")
    for tc_name in high_priority_tcs:
        tc_path = telecommand_pickle_path(tc_name)
        if not tc_path.exists():
            continue
        tc_raw = load_raw_pickle(tc_path).rename(columns={tc_name: "value"})
        tc_raw = _drop_tz(tc_raw)
        telecommand_frames[tc_name] = encode_telecommand(tc_raw, RESAMPLING_RULE).rename(columns={"value": tc_name})

wide = build_wide_dataset(channel_frames, telecommand_frames, RESAMPLING_RULE)
train_df, val_df, test_df = split_train_val_test(wide, CFG["test_data_split"], CFG["val_split"])

print(f"\nCombined shape: {wide.shape}")
print(f"Train: {len(train_df):,} rows | Val: {len(val_df):,} rows | Test: {len(test_df):,} rows")

channel_41: 15,381,169 raw samples -> 14,728,321 resampled rows | point events restored: 6


**Why local first, then Drive:** Colab's `/content` disk is fast but wiped the
moment the session ends; Drive (mounted over FUSE) survives, but is noticeably
slower to write to. Write once, fast, locally — then copy the finished (small)
set of files to Drive once, rather than writing every row over the network.

**File size heads-up:** even just the 6 lightweight channels for Mission1, at
30-second resolution over the ~7-year training window, comes out to roughly
7 million rows. CSV works and needs no extra dependency, but lands close to
1GB for the train split alone. If that's inconvenient, swap the three lines
below for `train_df.to_parquet(...)` etc. (needs `pip install pyarrow`) —
same data, a fraction of the size, faster both to write and to reload later.


In [ ]:
train_df.to_csv(OUTPUT_DIR_LOCAL / "train.csv")
val_df.to_csv(OUTPUT_DIR_LOCAL / "val.csv")
test_df.to_csv(OUTPUT_DIR_LOCAL / "test.csv")

metadata = {
    "mission": ACTIVE_MISSION,
    "target_channels": TARGET_CHANNELS,
    "resampling_rule_seconds": RESAMPLING_RULE.total_seconds(),
    "test_data_split": CFG["test_data_split"],
    "val_split": CFG["val_split"],
    "include_telecommands": INCLUDE_TELECOMMANDS,
    "point_events_restored_per_channel": restore_counts,
    "reference": "Kotowski et al., ESA-ADB (2025), Appendix C.3",
}
with open(OUTPUT_DIR_LOCAL / "preprocessing_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"Saved train/val/test + metadata locally to {OUTPUT_DIR_LOCAL.resolve()}")


# **11.1] PERSIST TO DRIVE**
This is the step that actually survives after the Colab session disconnects.
Run it once the local save above finishes successfully.


In [ ]:
import shutil

OUTPUT_DIR_DRIVE.mkdir(parents=True, exist_ok=True)
for fname in ["train.csv", "val.csv", "test.csv", "preprocessing_metadata.json"]:
    src = OUTPUT_DIR_LOCAL / fname
    if src.exists():
        shutil.copy(src, OUTPUT_DIR_DRIVE / fname)
        print(f"Copied {fname} -> {OUTPUT_DIR_DRIVE / fname}")
    else:
        print(f"Skipped {fname} (not found locally — did the save cell above run?)")

print(f"\nPersisted to Drive at: {OUTPUT_DIR_DRIVE}")


# **12. VALIDATE AGAINST REAL KNOWN EVENTS (PAPER TABLE 7 / ANOMALY_TYPES.CSV)**
This is the check that matters most: confirm real point-type events in the raw
data actually survive into the final combined output, not just the synthetic
toy example from section 7. Adjust the column-name guesses below if your
`anomaly_types.csv` uses different names — the printed columns from cell 5
tell you what to use.


In [ ]:
length_col_candidates = [c for c in anomaly_types_df.columns if "length" in c.lower()]
print(f"Guessed 'point vs subsequence' column: {length_col_candidates}")

if length_col_candidates:
    lcol = length_col_candidates[0]
    point_event_ids = anomaly_types_df.loc[
        anomaly_types_df[lcol].astype(str).str.contains("point", case=False, na=False), "ID"
    ].tolist()
    relevant = labels_df[labels_df["ID"].isin(point_event_ids) & labels_df["Channel"].isin(TARGET_CHANNELS)]
    print(f"Found {len(relevant)} point-type event(s) affecting our target channels in this split.\n")

    for _, row in relevant.iterrows():
        col = f"is_anomaly_{row['Channel']}"
        if col not in wide.columns:
            continue
        # allow a small tolerance window since the correction shifts a restored point
        # forward by up to one resampling step
        window = wide.loc[row["StartTime"] - RESAMPLING_RULE: row["EndTime"] + RESAMPLING_RULE, col]
        survived = (window > AnnotationLabel.NOMINAL).any()
        status = "OK - present" if survived else "MISSING"
        print(f"  event {row['ID']} on {row['Channel']} at {row['StartTime']}: {status}")
else:
    print("Could not auto-detect a point/subsequence column in anomaly_types.csv — "
          "inspect anomaly_types_df.columns above and adjust length_col_candidates manually.")


### Class balance sanity check

Quick sanity check on how much of each label survived per channel — useful for
noticing anything that looks obviously wrong (e.g. an entire channel showing 0%
anomalies when the metadata says it should have some) before moving forward.


In [ ]:
# raw counts alongside percentage -- with real anomaly rates under 2% (per the
# paper), a handful of point events out of millions of rows can round to
# "0.000%" and look like they're missing even when section 11 above confirms
# they survived. Counts make that unambiguous.
for channel_name in TARGET_CHANNELS:
    col = f"is_anomaly_{channel_name}"
    if col in wide.columns:
        counts = wide[col].value_counts().sort_index()
        pct = wide[col].value_counts(normalize=True).sort_index() * 100
        parts = [f"label {int(k)} = {counts[k]:,} rows ({pct[k]:.5f}%)" for k in counts.index]
        print(f"{channel_name}: " + ", ".join(parts))
